Run But need to mannual cancel the cell, otherwise it keeps listening

In [ ]:
%pip install google-cloud-pubsub python-dotenv

  Using cached google_cloud_pubsub-2.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached google_api_core-2.30.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached grpc_google_iam_v1-0.14.3-py3-none-any.whl.metadata (9.2 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached google_cloud_pubsub-2.36.0-py3-none-any.whl (323 kB)
Using cached google_api_core-2.30.0-py3-none-any.whl (173 kB)
Using cached grpc_google_iam_v1-0.14.3-py3-none-any.whl (32 kB)
Using cached opentelemetry_api-1.40.0-py3-none-any.whl (68 kB)
Using cached importlib_metadata-8.7.1-py3-none-any.whl (27 kB)
Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl (141 kB)
Using cached o

In [ ]:

from google.cloud import pubsub_v1
import json
import os

# 1. Configuration from your .env
project_id = os.getenv("GCP_PROJECT_ID")
subscription_id = "bitcoin-price-sub" # The name from your Terraform

# 2. Initialize the Subscriber
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

print(f"🚀 Listening for Bitcoin data on {subscription_id}...")

def callback(message: pubsub_v1.subscriber.message.Message) -> None:
    """
    Process the incoming Bitcoin price message.
    """
    # Parse the data
    payload = json.loads(message.data.decode("utf-8"))
    
    # Senior View: Extract key metrics
    asset = payload['data']['asset']
    price_aud = payload['data']['price_aud']
    timestamp = payload['metadata']['ingested_at']
    
    print(f"✅ Received {asset.upper()}: ${price_aud:,.2f} AUD | Ingested: {timestamp}")
    
    # CRITICAL: Acknowledge the message so it's removed from the 'Unacked' graph
    message.ack()

# 3. Start the "Pull" flow
# We limit to 5 messages for this notebook test
streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)

try:
    # Keep the notebook cell alive for 30 seconds to catch messages
    streaming_pull_future.result(timeout=30)
except Exception as e:
    streaming_pull_future.cancel()
    print(f"\n⏹️ Stopped listening. (Note: {e})")

🚀 Listening for Bitcoin data on bitcoin-price-sub...


KeyboardInterrupt: 

Run But it will stop by itself

In [ ]:
from google.cloud import pubsub_v1
import os

# 1. Setup
project_id = os.getenv("GCP_PROJECT_ID")
subscription_id = "bitcoin-price-sub"
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

print(f"📥 Pulling a batch of messages from {subscription_id}...")

# 2. THE SYNC PULL (The "Senior" Test Choice)
# 'max_messages' tells GCP: "Give me up to 5, then stop talking to me."
response = subscriber.pull(
    request={
        "subscription": subscription_path, 
        "max_messages": 5
    }
)

if not response.received_messages:
    print("📭 No messages waiting in the mailbox.")
else:
    for received_message in response.received_messages:
        msg = received_message.message
        data = json.loads(msg.data.decode("utf-8"))
        
        asset = data['data']['asset']
        price = data['data']['price_aud']
        
        print(f"✅ Picked up {asset.upper()}: ${price:,.2f} AUD")
        
        # 3. ACKNOWLEDGE: "I've got it, don't send it again."
        subscriber.acknowledge(
            request={
                "subscription": subscription_path,
                "ack_ids": [received_message.ack_id],
            }
        )

print("\n🏁 Cell finished. Mailbox is clear.")

📥 Pulling a batch of messages from bitcoin-price-sub...
✅ Picked up ETHEREUM: $3,020.81 AUD

🏁 Cell finished. Mailbox is clear.


✅ Received BITCOIN: $98,821.00 AUD | Ingested: 2026-03-30T12:14:41.687356+00:00
✅ Received BITCOIN: $98,821.00 AUD | Ingested: 2026-03-30T12:15:08.691054+00:00
✅ Received ETHEREUM: $3,020.81 AUD | Ingested: 2026-03-30T12:15:08.838989+00:00
✅ Received ETHEREUM: $3,020.95 AUD | Ingested: 2026-03-30T12:16:09.236248+00:00
✅ Received BITCOIN: $98,864.00 AUD | Ingested: 2026-03-30T12:16:09.200406+00:00
